In [ ]:
import json
import pandas as pd
from pathlib import Path
from collections import Counter

FILES = {
    "usernames": Path("../dataset_normalized/usernames_normalized.jsonl"),
    "phone_numbers": Path("../dataset_normalized/numbers_normalized.jsonl")
}

def analizza_details(path):
    campi = {
        "nft_transfer": Counter(),
        "nft_purchase": Counter()
    }

    totali = Counter()

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            record = json.loads(line)
            tipo = record.get("action_type")

            if tipo not in campi:
                continue

            details = record.get("details") or {}
            if not isinstance(details, dict):
                continue

            totali[tipo] += 1

            for key, value in details.items():
                if value is not None:
                    campi[tipo][key] += 1

    risultati = []

    for tipo, counter in campi.items():
        for campo, count in counter.items():
            risultati.append({
                "action_type": tipo,
                "campo": campo,
                "presenti": count,
                "totale": totali[tipo],
                "percentuale": round(
                    100 * count / totali[tipo], 2
                ) if totali[tipo] else 0
            })

    return pd.DataFrame(risultati)


for layer, path in FILES.items():
    print(f"\n{'=' * 60}")
    print(layer.upper())
    print(f"{'=' * 60}")

    df = analizza_details(path)

    display(
        df.sort_values(
            ["action_type", "percentuale"],
            ascending=[True, False]
        )
    )


USERNAMES


,action_type,campo,presenti,totale,percentuale
20,nft_purchase,nft_collection,78371,78371,100.00
21,nft_purchase,nft_item,78371,78371,100.00
22,nft_purchase,nft_item_index,78371,78371,100.00
23,nft_purchase,old_owner,78371,78371,100.00
24,nft_purchase,new_owner,78371,78371,100.00
25,nft_purchase,is_purchase,78371,78371,100.00
26,nft_purchase,price,78371,78371,100.00
27,nft_purchase,query_id,78371,78371,100.00
28,nft_purchase,response_destination,78371,78371,100.00
29,nft_purchase,forward_payload,78371,78371,100.00



PHONE_NUMBERS


,action_type,campo,presenti,totale,percentuale
19,nft_purchase,nft_collection,47311,47311,100.00
20,nft_purchase,nft_item,47311,47311,100.00
21,nft_purchase,nft_item_index,47311,47311,100.00
22,nft_purchase,old_owner,47311,47311,100.00
23,nft_purchase,new_owner,47311,47311,100.00
24,nft_purchase,is_purchase,47311,47311,100.00
25,nft_purchase,price,47311,47311,100.00
26,nft_purchase,query_id,47311,47311,100.00
27,nft_purchase,response_destination,47311,47311,100.00
28,nft_purchase,forward_payload,47311,47311,100.00


In [2]:
from collections import Counter

def controlla_wallet(path):
    risultati = Counter()

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            r = json.loads(line)
            tipo = r.get("action_type")

            if tipo not in {"nft_transfer", "nft_purchase"}:
                continue

            d = r.get("details") or {}
            if not isinstance(d, dict):
                continue

            risultati[(tipo, "totale")] += 1

            if d.get("new_owner"):
                risultati[(tipo, "new_owner")] += 1

            if d.get("old_owner"):
                risultati[(tipo, "old_owner")] += 1

            if d.get("real_old_owner"):
                risultati[(tipo, "real_old_owner")] += 1

            if d.get("new_owner") and d.get("real_old_owner"):
                risultati[(tipo, "sender_recipient_reali")] += 1

            if d.get("new_owner") and (
                d.get("real_old_owner") or d.get("old_owner")
            ):
                risultati[(tipo, "sender_recipient_candidati")] += 1

    return pd.DataFrame([
        {
            "action_type": tipo,
            "metrica": metrica,
            "conteggio": valore
        }
        for (tipo, metrica), valore in risultati.items()
    ])


for layer, path in FILES.items():
    print(f"\n{layer.upper()}")
    display(controlla_wallet(path))


USERNAMES


,action_type,metrica,conteggio
0,nft_transfer,totale,2500782
1,nft_transfer,new_owner,2500782
2,nft_transfer,old_owner,1945477
3,nft_transfer,sender_recipient_candidati,1945477
4,nft_purchase,totale,78371
5,nft_purchase,new_owner,78371
6,nft_purchase,old_owner,78371
7,nft_purchase,real_old_owner,78371
8,nft_purchase,sender_recipient_reali,78371
9,nft_purchase,sender_recipient_candidati,78371



PHONE_NUMBERS


,action_type,metrica,conteggio
0,nft_transfer,totale,789835
1,nft_transfer,new_owner,789835
2,nft_transfer,old_owner,653249
3,nft_transfer,sender_recipient_candidati,653249
4,nft_purchase,totale,47311
5,nft_purchase,new_owner,47311
6,nft_purchase,old_owner,47311
7,nft_purchase,real_old_owner,47311
8,nft_purchase,sender_recipient_reali,47311
9,nft_purchase,sender_recipient_candidati,47311


In [3]:
from pprint import pprint

for layer, path in FILES.items():
    print(f"\n{'=' * 60}")
    print(layer.upper())
    print(f"{'=' * 60}")

    esempi = {
        "nft_transfer": [],
        "nft_purchase": []
    }

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if all(len(v) >= 3 for v in esempi.values()):
                break

            r = json.loads(line)
            tipo = r.get("action_type")

            if tipo in esempi and len(esempi[tipo]) < 3:
                esempi[tipo].append(r)

    for tipo, records in esempi.items():
        print(f"\n--- {tipo.upper()} ---")

        for r in records:
            print("\nNFT:", r.get("nft"))
            print("Timestamp:", r.get("timestamp"))
            print("Trace:", r.get("trace_id"))
            pprint(r.get("details"), sort_dicts=False)


USERNAMES

--- NFT_TRANSFER ---

NFT: 0:9c81d90cd3c57df4e0593e3493497894021c392cd011b88dcbcae377a865a16a
Timestamp: 2025-08-12T08:20:45Z
Trace: fUJfAk3EScZHAsS/+unO2heesled95K+2/9MDfjEkOY=
{'nft_collection': '0:80D78A35F955A14B679FAA887FF4CD5BFC0F43B4A4EEA2A7E6927F3701B273C2',
 'nft_item': '0:9C81D90CD3C57DF4E0593E3493497894021C392CD011B88DCBCAE377A865A16A',
 'nft_item_index': '1049143216575933698300671218993488121127102958821723117678060246960963584',
 'new_owner': '0:E839FE251D2C249E8E8EA56CE7AE98BC7B80CCB8FDD7164D6277B0E49088F7B1',
 'is_purchase': True,
 'price': '10000000000',
 'query_id': '60384214000001',
 'response_destination': None,
 'custom_payload': None,
 'forward_payload': 'te6cckEBAQEAEAAAGzgSfeFQJUC+QAaJG+yoFKgzWw==',
 'forward_amount': None,
 'comment': None,
 'is_encrypted_comment': False,
 'marketplace': None,
 'real_old_owner': None,
 'marketplace_address': None,
 'payout_amount': '8479108578',
 'payout_address': '0:0B6210F295119197F214ABB61C9731340AFF4D914CF9FE7785

In [4]:
import json
import pandas as pd
from pathlib import Path
from collections import defaultdict

FILES = {
    "usernames": Path("dataset_normalized/usernames_normalized.jsonl"),
    "phone_numbers": Path("dataset_normalized/numbers_normalized.jsonl")
}

def verifica_overlap_actions(path):
    # Per ogni (NFT, trace_id), registra i tipi presenti
    eventi = defaultdict(set)

    for line in path.open("r", encoding="utf-8"):
        if not line.strip():
            continue

        r = json.loads(line)

        tipo = r.get("action_type")
        if tipo not in {"nft_transfer", "nft_purchase"}:
            continue

        nft = r.get("nft")
        trace = r.get("trace_id")

        if not nft or not trace:
            continue

        eventi[(nft.lower(), trace)].add(tipo)

    solo_transfer = 0
    solo_purchase = 0
    entrambi = 0

    for tipi in eventi.values():
        if tipi == {"nft_transfer"}:
            solo_transfer += 1
        elif tipi == {"nft_purchase"}:
            solo_purchase += 1
        elif tipi == {"nft_transfer", "nft_purchase"}:
            entrambi += 1

    return {
        "solo_transfer": solo_transfer,
        "solo_purchase": solo_purchase,
        "entrambi": entrambi,
        "totale_gruppi": len(eventi)
    }

risultati = []

for layer, path in FILES.items():
    print(f"Analizzo {layer}...")
    stats = verifica_overlap_actions(path)
    risultati.append({"layer": layer, **stats})

display(pd.DataFrame(risultati))

Analizzo usernames...
Analizzo phone_numbers...


,layer,solo_transfer,solo_purchase,entrambi,totale_gruppi
0,usernames,2426173,78371,0,2504544
1,phone_numbers,767682,47311,0,814993


In [5]:
import json
import pandas as pd
from pathlib import Path
from collections import defaultdict

FILES = {
    "usernames": Path("dataset_normalized/usernames_normalized.jsonl"),
    "phone_numbers": Path("dataset_normalized/numbers_normalized.jsonl")
}

def analizza_purchase_transfer(path, window_seconds=60):

    # Conserviamo soltanto i purchase
    purchases = defaultdict(list)

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            r = json.loads(line)

            if r.get("action_type") != "nft_purchase":
                continue

            details = r.get("details") or {}

            purchases[str(r["nft"]).lower()].append({
                "timestamp": pd.Timestamp(r["timestamp"]).timestamp(),
                "new_owner": str(details.get("new_owner", "")).lower(),
                "trace_id": r.get("trace_id")
            })

    stats = {
        "purchase_totali": sum(len(v) for v in purchases.values()),
        "purchase_con_transfer": set(),
        "stesso_timestamp": set(),
        "stesso_destinatario": set()
    }

    # Cerchiamo i transfer corrispondenti
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            r = json.loads(line)

            if r.get("action_type") != "nft_transfer":
                continue

            nft = str(r.get("nft", "")).lower()

            if nft not in purchases:
                continue

            ts = pd.Timestamp(r["timestamp"]).timestamp()
            details = r.get("details") or {}
            new_owner = str(details.get("new_owner", "")).lower()

            for i, p in enumerate(purchases[nft]):
                delta = abs(ts - p["timestamp"])

                if delta <= window_seconds:
                    key = (nft, i)

                    stats["purchase_con_transfer"].add(key)

                    if delta == 0:
                        stats["stesso_timestamp"].add(key)

                    if new_owner == p["new_owner"]:
                        stats["stesso_destinatario"].add(key)

    return {
        "purchase_totali": stats["purchase_totali"],
        "purchase_con_transfer_60s": len(stats["purchase_con_transfer"]),
        "stesso_timestamp": len(stats["stesso_timestamp"]),
        "stesso_destinatario_60s": len(stats["stesso_destinatario"])
    }

risultati = []

for layer, path in FILES.items():
    print(f"Analizzo {layer}...")

    result = analizza_purchase_transfer(path)

    risultati.append({
        "layer": layer,
        **result
    })

display(pd.DataFrame(risultati))

Analizzo usernames...
Analizzo phone_numbers...


,layer,purchase_totali,purchase_con_transfer_60s,stesso_timestamp,stesso_destinatario_60s
0,usernames,78371,6015,0,133
1,phone_numbers,47311,3304,3,13


In [6]:
import json
import pandas as pd
from pathlib import Path
from collections import Counter

FILES = {
    "usernames": Path("dataset_normalized/usernames_normalized.jsonl"),
    "phone_numbers": Path("dataset_normalized/numbers_normalized.jsonl")
}

OUT = Path("cleaning/old_owner_check")
OUT.mkdir(parents=True, exist_ok=True)

def norm_addr(value):
    return str(value).strip().lower() if value else None

In [7]:
def controlla_old_owner(path, layer):

    # Primo trasferimento/acquisto registrato per NFT
    primo_evento = {}

    # Prima scansione
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            r = json.loads(line)

            nft = norm_addr(r.get("nft"))
            if not nft:
                continue

            ts = pd.Timestamp(r["timestamp"]).timestamp()
            slt = int(r.get("start_lt") or 0)

            ordine = (ts, slt)

            if nft not in primo_evento or ordine < primo_evento[nft]:
                primo_evento[nft] = ordine

    stats = Counter()
    esempi = []

    # Seconda scansione
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            r = json.loads(line)

            if r.get("action_type") != "nft_transfer":
                continue

            d = r.get("details") or {}

            if d.get("old_owner"):
                continue

            nft = norm_addr(r.get("nft"))
            ts = pd.Timestamp(r["timestamp"]).timestamp()
            slt = int(r.get("start_lt") or 0)

            ordine = (ts, slt)

            stats["senza_old_owner"] += 1

            if ordine == primo_evento.get(nft):
                stats["primo_trasferimento"] += 1
                stato = "FIRST_TRANSFER"
            else:
                stats["trasferimento_successivo"] += 1
                stato = "LATER_TRANSFER"

            if len(esempi) < 100:
                esempi.append({
                    "nft": nft,
                    "timestamp": r["timestamp"],
                    "action_type": r["action_type"],
                    "new_owner": d.get("new_owner"),
                    "status": stato
                })

    df = pd.DataFrame(
        [{"metrica": k, "conteggio": v} for k, v in stats.items()]
    )

    pd.DataFrame(esempi).to_csv(
        OUT / f"examples_{layer}.csv",
        index=False
    )

    return df

In [8]:
for layer, path in FILES.items():
    print(f"\n{'=' * 50}")
    print(layer.upper())
    print(f"{'=' * 50}")

    result = controlla_old_owner(path, layer)

    display(result)


USERNAMES


,metrica,conteggio
0,senza_old_owner,555305
1,primo_trasferimento,551699
2,trasferimento_successivo,3606



PHONE_NUMBERS


,metrica,conteggio
0,senza_old_owner,136586
1,primo_trasferimento,136566
2,trasferimento_successivo,20
